In [1]:
from pathlib import Path

def load_api_key_from_file(path: Path) -> str:
    if not path.exists():
        raise FileNotFoundError(f"API key file not found: {path}")
    key = path.read_text(encoding="utf-8").strip()
    if not key:
        raise ValueError(f"API key file is empty: {path}")
    return key

key_path = Path.home() / "env" / "openai_secret_key.txt"

OPENAI_API_KEY = load_api_key_from_file(key_path)

import os
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

In [2]:
import json
from datasets import load_dataset
from openai import OpenAI
from datetime import datetime
dataset_test = load_dataset("mkieffer/MEDEC", split="test")
client = OpenAI()

/home/heewon/miniconda3/envs/nlp-project/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
dataset_test[0]

{'text_id': 'ms-test-0',
 'text': "A 29-year-old internal medicine resident presents to the emergency department with complaints of fevers, diarrhea, abdominal pain, and skin rash for 2 days. He feels fatigued and has lost his appetite. On further questioning, he says that he returned from his missionary trip to Brazil last week. He is excited as he talks about his trip. Besides a worthy clinical experience, he also enjoyed local outdoor activities, like swimming and rafting. His past medical history is insignificant. The blood pressure is 120/70 mm Hg, the pulse is 100/min, and the temperature is 38.3 C (100.9 F). On examination, there is a rash on the legs. Patient's symptoms are suspected to be due to hepatitis A. The rest of the examination is normal.",
 'sentences': "0 | A 29-year-old internal medicine resident presents to the emergency department with complaints of fevers, diarrhea, abdominal pain, and skin rash for 2 days.\n1 | He feels fatigued and has lost his appetite.\n2 | O

In [4]:
def load_indices(path: str):
    return json.loads(Path(path).read_text(encoding="utf-8"))["indices"]

loaded_indices = load_indices("sampled_test_indices.json")
subset_test = dataset_test.select(loaded_indices)

In [6]:
# system = """
# The following is a medical narrative about a patient. You are a skilled medical doctor reviewing the clinical text. The text is either correct or contains one error.
# The text has a sentence per line. Each line starts with the sentence ID, followed by a pipe character then the sentence to check. Check every sentence of the text.
# If the text is correct return the following output: CORRECT. If the text has a medical error, return the sentence id of the sentence containing the error,
# followed by a space, and a corrected version of the sentence.
# """

system = """
The following is a medical narrative about a patient. You are a skilled medical doctor reviewing the clinical text. The text is either correct or contains one error.
The text has one sentence per line. Each line starts with the sentence ID, followed by a pipe character then the sentence to check. Check every sentence of the text.
If the text is correct return the following output: CORRECT. If the text has a medical error related to treatment, management, cause, or diagnosis, return the sentence id of the sentence containing the error, followed by a space, and then a corrected version of the sentence.
Finding and correcting the error requires medical knowledge and reasoning.
"""

def get_text_id(item: dict, fallback_i: int) -> str:
    # Try common keys that might store a text/document identifier.
    # If none exist, fall back to a deterministic id based on the index.
    for k in ["text_id", "textid", "id", "uid", "doc_id", "note_id"]:
        if k in item and item[k] is not None:
            return str(item[k])
    return f"idx-{fallback_i}"

def escape_for_double_quotes(s: str) -> str:
    # Escape backslashes and double quotes so the output stays valid when wrapped in "...".
    return s.replace("\\", "\\\\").replace('"', '\\"')

def parse_model_output(output_text: str):
    """
    Parse the model output into the submission fields.

    Expected outputs:
      - "CORRECT." (or variants like "CORRECT", "correct.")  -> no error
      - "<sentence_id> <corrected sentence>"                 -> error
    Returns: (error_flag:int, error_sentence_id:int, corrected_sentence:str|None)
    """
    t = (output_text or "").strip()

    # Treat CORRECT (case-insensitive) with an optional trailing period as "no error"
    if t.upper().rstrip(".") == "CORRECT":
        return 0, -1, None

    # Otherwise expect: "<sid> <corrected sentence...>"
    parts = t.split(None, 1)  # split into [sid, rest] (rest can contain spaces)
    if not parts:
        # Defensive fallback for unexpected empty output
        return 0, -1, None

    # Parse sentence id
    try:
        sid = int(parts[0])
    except ValueError:
        # Defensive fallback for unexpected format
        return 0, -1, None

    corrected = parts[1].strip() if len(parts) > 1 else ""
    if not corrected:
        # If only sid is returned, avoid empty corrected sentence
        corrected = "NA"

    return 1, sid, corrected

# ===== main =====

GPT5_MODEL = "gpt-5"
GPT5_2_MODEL = "gpt-5.2"
GPT5_2_PRO_MODEL = "gpt-5.2-pro"
GPT4_1_MODEL = "gpt-4.1"
O1_MODEL = "o1"

model = O1_MODEL

out_dir = Path("outputs")
out_dir.mkdir(parents=True, exist_ok=True)
out_path = out_dir / f"medec-ms_{model}_results_{datetime.now().strftime('%Y%m%d_%H%M%S')}.txt"

num_examples = len(subset_test)  # Change to len(subset_test) to run on the full test split
with out_path.open("w", encoding="utf-8") as f:
    for i in range(num_examples):
        item = subset_test[i]
        text_id = item["text_id"]

        messages = [
            {"role": "system", "content": system},
            {"role": "user", "content": item["sentences"]},
        ]

        if model == GPT5_MODEL:
            response = client.responses.create(
                model=model,
                reasoning={"effort": "low"},
                input=messages
            )
        elif model == GPT5_2_MODEL:
            response = client.responses.create(
                model=model,
                reasoning={"effort": "low"},
                input=messages
            )
        elif model == GPT5_2_PRO_MODEL:
            response = client.responses.create(
                model=model,
                reasoning={"effort": "high"},
                input=messages
            )
        elif model == O1_MODEL:
            response = client.responses.create(
                model=model,
                reasoning={"effort": "medium"},
                input=messages
            )
        elif model == GPT4_1_MODEL:
            response = client.responses.create(
                model=model,
                input=messages
            )
        else:
            raise ValueError(f"Unsupported model: {model}")
        output_text = response.output_text
        flag, sid, corrected = parse_model_output(output_text)

        # Write one line per example in the required format:
        # [Text ID] [Error Flag] [Error sentence ID or -1] [Corrected sentence or NA]
        if flag == 0:
            line = f"{text_id} 0 -1 NA"
        else:
            corrected_escaped = escape_for_double_quotes(corrected)
            line = f'{text_id} 1 {sid} "{corrected_escaped}"'

        f.write(line + "\n")
        print(line)
        print("text_id:", item["text_id"])
        print("error_sentence_id:", item["error_sentence_id"], "True Positive:", sid == item["error_sentence_id"])

print("Saved to:", out_path)


ms-test-17 1 4 "Mycobacterium tuberculosis is suspected."
text_id: ms-test-17
error_sentence_id: 4 True Positive: True
ms-test-22 0 -1 NA
text_id: ms-test-22
error_sentence_id: -1 True Positive: True
ms-test-29 1 21 "Diffuse cutaneous systemic scleroderma is suspected."
text_id: ms-test-29
error_sentence_id: 21 True Positive: True
ms-test-43 1 6 "Hemoglobin 14 g/dL"
text_id: ms-test-43
error_sentence_id: 4 True Positive: False
ms-test-45 1 4 "The patient is diagnosed with polyarteritis nodosa."
text_id: ms-test-45
error_sentence_id: 4 True Positive: True
ms-test-53 1 13 "Patient is diagnosed with Zollinger-Ellison syndrome (gastrinoma)."
text_id: ms-test-53
error_sentence_id: 13 True Positive: True
ms-test-67 1 15 "The patient is diagnosed with normal pressure hydrocephalus after further evaluation reveals the triad of cognitive decline, gait disturbance, and urinary incontinence."
text_id: ms-test-67
error_sentence_id: 15 True Positive: True
ms-test-68 1 17 "The patient is diagnosed w